# Coverage Evaluator Benchmark

Benchmarks the **final public `CoverageEvaluator` API** only — both modes, same
metric name `coverage`, contract `context + output` (no `input`):

- **`mode="g_eval"`** — one judge call identifies + classifies source items.
- **`mode="dag"`** — Stage 1 extracts ~10 semantically consolidated source items
  from the **context only**; Stage 2 classifies that fixed set against the output.
  Two judge calls normally; safety batching only above 20 items.

There is **no** notebook-local DAG implementation and **no** deprecated
`SourceCoverageEvaluator` here — the notebook runs exactly what users run.

This notebook answers: how fast is each mode, how many judge calls, how stable are
scores and denominators across repeated runs, which mode is closer to ground truth,
and whether the new `~10` consolidated-item DAG design preserves DAG quality.

> Live LLM cells are **not** auto-run. Run them manually when you want fresh
> numbers; any output shown below a benchmark cell should be regenerated after
> editing (stale pre-refactor outputs have been cleared).

## 1. Setup

In [ ]:
import time

import numpy as np
import pandas as pd

from idp_eval import (
    CoverageEvaluator,
    EvaluationCase,
    EvaluationFramework,
    create_judge,
)

MAX_CONCURRENCY = 4

# Reuse the existing session config. create_judge reads credentials from the
# environment; nothing secret is printed in this notebook.
judge = create_judge(verify_ssl=False)

g_eval_evaluator = CoverageEvaluator(judge, mode="g_eval", verbose=False)
dag_evaluator = CoverageEvaluator(judge, mode="dag", verbose=False)

g_eval_framework = EvaluationFramework(judge=judge, evaluators=[g_eval_evaluator])
dag_framework = EvaluationFramework(judge=judge, evaluators=[dag_evaluator])

# Sanity: both carry the same public metric name, in their own mode.
assert g_eval_evaluator.name == dag_evaluator.name == "coverage"
assert (g_eval_evaluator._mode, dag_evaluator._mode) == ("g_eval", "dag")
print("frameworks ready:", g_eval_evaluator._mode, "vs", dag_evaluator._mode)

## 2. Load Ground Truth Dataset

In [ ]:
GT_COLUMN = "gt_source_coverage"  # ground-truth coverage column in the golden set

df = pd.read_csv("golden_set_augmented_tagged.csv").fillna("")

if GT_COLUMN not in df.columns:
    raise ValueError(
        f"Ground-truth column {GT_COLUMN!r} not found. "
        f"Available columns: {list(df.columns)}"
    )

# GT -> numeric on the 0-1 scale, normalizing percentages (>1) exactly once.
gt_raw = pd.to_numeric(df[GT_COLUMN], errors="coerce")
gt_series = gt_raw.where(gt_raw <= 1.0, gt_raw / 100.0)

print(f"rows: {len(df)}")
print(f"GT column: {GT_COLUMN}")
print(f"GT min/max/mean: {gt_series.min():.3f} / {gt_series.max():.3f} / {gt_series.mean():.3f}")

missing_gt = int(gt_raw.isna().sum())
if missing_gt:
    print(f"rows with non-numeric / missing GT: {missing_gt}")

if "epic_key" in df.columns:
    dup_case_ids = int(df["epic_key"].astype(str).duplicated().sum())
    print(f"duplicate case_id (epic_key) count: {dup_case_ids}")

## 3. Build Evaluation Cases

One `EvaluationCase` per row (`context + output` only — GT is **never** sent to the
evaluator). `case_id` can repeat, so every row also gets a unique `benchmark_id`
(`f"{case_id}:{row_index}"`) used for all joins and comparisons.

In [ ]:
cases = []
benchmark_ids = []
gt_by_bid = {}

for i, row in df.iterrows():
    case_id = str(row.get("epic_key", i))
    benchmark_id = f"{case_id}:{i}"
    cases.append(
        EvaluationCase(
            context={
                "theme_business_needs": row.get("theme_businessNeeds", ""),
                "theme_description": row.get("theme_description", ""),
            },
            output={
                "epic_description": row.get("epic_description", ""),
                "epic_success_criteria": row.get("epic_successCriteria", ""),
            },
            case_id=case_id,
        )
    )
    benchmark_ids.append(benchmark_id)
    gt_val = gt_series.iloc[i]
    gt_by_bid[benchmark_id] = None if pd.isna(gt_val) else float(gt_val)

assert len(set(benchmark_ids)) == len(benchmark_ids)  # duplicate-safe join keys
print(f"built {len(cases)} cases; {len(set(benchmark_ids))} unique benchmark_ids")


def rows_from_results(results, fallback_mode):
    """Flatten a_evaluate_many results into benchmark rows (details read safely)."""
    out = []
    for bid, case, result_map in zip(benchmark_ids, cases, results):
        result = result_map["coverage"]
        d = result.details or {}
        gt = gt_by_bid.get(bid)
        score = result.score
        signed = (score - gt) if (gt is not None and score is not None) else None
        out.append({
            "benchmark_id": bid,
            "case_id": case.case_id,
            "mode": d.get("mode", fallback_mode),
            "score": score,
            "label": result.label,
            "final_item_count": d.get("final_item_count"),
            "covered_count": d.get("covered_count"),
            "partial_count": d.get("partial_count"),
            "missing_count": d.get("missing_count"),
            "judge_call_count": d.get("judge_call_count"),
            "batch_count": d.get("batch_count"),
            "extract_ms": d.get("extract_ms"),
            "classify_ms": d.get("classify_ms"),
            "total_ms": d.get("total_ms"),
            "gt_score": gt,
            "signed_error": signed,
            "abs_error": abs(signed) if signed is not None else None,
        })
    return out

## 4. G-Eval Async Benchmark

`CoverageEvaluator(mode="g_eval")` over the whole dataset via the framework async
API. Normally `judge_call_count == 1` per case.

In [ ]:
start = time.perf_counter()
g_eval_results = await g_eval_framework.a_evaluate_many(cases, max_concurrency=MAX_CONCURRENCY)
g_eval_wall_s = time.perf_counter() - start

df_g_eval = pd.DataFrame(rows_from_results(g_eval_results, "g_eval"))

n_one = int((df_g_eval["judge_call_count"] == 1).sum())
print(f"g_eval wall time: {g_eval_wall_s:.2f}s")
print(f"judge_call_count == 1 on {n_one}/{len(df_g_eval)} cases")
display(df_g_eval)

## 5. DAG Async Benchmark

`CoverageEvaluator(mode="dag")` over the same cases, judge, ordering and
concurrency. Normal (≤20-item) cases show `judge_call_count == 2` and
`batch_count == 1`; genuine large extractions (>20 items) may trigger safety
batching, so we **report** the distribution rather than asserting it.

In [ ]:
start = time.perf_counter()
dag_results = await dag_framework.a_evaluate_many(cases, max_concurrency=MAX_CONCURRENCY)
dag_wall_s = time.perf_counter() - start

df_dag = pd.DataFrame(rows_from_results(dag_results, "dag"))

n_two = int((df_dag["judge_call_count"] == 2).sum())
n_more = int((df_dag["judge_call_count"] > 2).sum())
n_over20 = int((df_dag["final_item_count"] > 20).sum())
print(f"dag wall time: {dag_wall_s:.2f}s")
print(f"judge_call_count == 2: {n_two} cases")
print(f"judge_call_count > 2 (safety batching): {n_more} cases")
print(f"final_item_count > 20: {n_over20} cases")
display(df_dag)

### 5a. DAG item-count analysis

Does the new `~10` semantic consolidation target behave as intended? (Exactly 10 is
**not** required — we care about approximate consolidation, and >10 is allowed.)

In [ ]:
ic = df_dag["final_item_count"].dropna()
dag_item_count_summary = pd.DataFrame([{
    "min": int(ic.min()) if len(ic) else None,
    "median": ic.median() if len(ic) else None,
    "mean": round(ic.mean(), 2) if len(ic) else None,
    "max": int(ic.max()) if len(ic) else None,
    "cases_lt_8": int((ic < 8).sum()),
    "cases_8_to_12": int(((ic >= 8) & (ic <= 12)).sum()),
    "cases_13_to_20": int(((ic >= 13) & (ic <= 20)).sum()),
    "cases_gt_20": int((ic > 20).sum()),
}])
display(dag_item_count_summary)

### 5b. Safety batching analysis

Verifies the production timeout safeguard during this benchmark: safety batching
should occur only when a case genuinely extracts more than 20 items.

In [ ]:
over20 = df_dag[df_dag["final_item_count"] > 20]
batched = df_dag[df_dag["judge_call_count"] > 2]
if len(over20) == 0 and len(batched) == 0:
    print("No safety batching was triggered.")
else:
    print(f"{len(batched)} case(s) used >2 judge calls; {len(over20)} case(s) exceeded 20 items.")
    display(
        df_dag.loc[
            (df_dag["final_item_count"] > 20) | (df_dag["judge_call_count"] > 2),
            ["benchmark_id", "final_item_count", "batch_count", "judge_call_count"],
        ]
    )

## 6. Performance Comparison

In [ ]:
def _perf_row(arch, dfm, wall):
    lat = dfm["total_ms"].dropna()
    ic = dfm["final_item_count"].dropna()
    return {
        "architecture": arch,
        "cases": len(dfm),
        "wall_time_s": round(wall, 2),
        "mean_case_latency_ms": round(lat.mean(), 1) if len(lat) else None,
        "median_case_latency_ms": round(lat.median(), 1) if len(lat) else None,
        "total_judge_calls": int(dfm["judge_call_count"].fillna(0).sum()),
        "mean_judge_calls_per_case": round(dfm["judge_call_count"].mean(), 2),
        "mean_item_count": round(ic.mean(), 2) if len(ic) else None,
        "median_item_count": ic.median() if len(ic) else None,
        "max_item_count": int(ic.max()) if len(ic) else None,
        "cases_over_20_items": int((dfm["final_item_count"] > 20).sum()),
        "mean_score": round(dfm["score"].mean(), 4),
    }


df_performance_summary = pd.DataFrame([
    _perf_row("g_eval", df_g_eval, g_eval_wall_s),
    _perf_row("dag", df_dag, dag_wall_s),
])
display(df_performance_summary)

perf = df_performance_summary.set_index("architecture")
dag_vs_g_eval_wall_time_ratio = perf.loc["dag", "wall_time_s"] / perf.loc["g_eval", "wall_time_s"]
dag_vs_g_eval_call_ratio = perf.loc["dag", "total_judge_calls"] / perf.loc["g_eval", "total_judge_calls"]
print(f"dag_vs_g_eval_wall_time_ratio: {dag_vs_g_eval_wall_time_ratio:.2f}")
print(f"dag_vs_g_eval_call_ratio: {dag_vs_g_eval_call_ratio:.2f}")

**Operational reading only** (no quality winner here): a ratio > 1 means DAG is
slower / makes more judge calls than G-Eval, which is expected since DAG is
two-stage. Quality is decided in the GT-closeness sections below.

## 7. Stability Experiment

Run the full dataset `N_RUNS` times for **each** mode (no serial one-call variant).
Rows are keyed by `benchmark_id` because `case_id` can repeat.

In [ ]:
N_RUNS = 3


async def _run_once(framework, arch, run_number):
    start = time.perf_counter()
    results = await framework.a_evaluate_many(cases, max_concurrency=MAX_CONCURRENCY)
    wall = time.perf_counter() - start
    rows = []
    for bid, case, result_map in zip(benchmark_ids, cases, results):
        r = result_map["coverage"]
        d = r.details or {}
        gt = gt_by_bid.get(bid)
        signed = (r.score - gt) if (gt is not None and r.score is not None) else None
        rows.append({
            "benchmark_id": bid, "case_id": case.case_id, "run": run_number,
            "architecture": arch, "score": r.score,
            "item_count": d.get("final_item_count"),
            "judge_calls": d.get("judge_call_count"), "total_ms": d.get("total_ms"),
            "gt_score": gt, "signed_error": signed,
            "abs_error": abs(signed) if signed is not None else None,
        })
    return rows, wall


run_rows, run_summary_rows, failures = [], [], []
for run in range(1, N_RUNS + 1):
    for arch, framework in [("g_eval", g_eval_framework), ("dag", dag_framework)]:
        try:
            rows, wall = await _run_once(framework, arch, run)
        except Exception as exc:  # no retry; record and continue
            failures.append({"architecture": arch, "run": run, "error": f"{type(exc).__name__}: {exc}"})
            print(f"{arch} run {run} FAILED: {type(exc).__name__}: {exc}")
            continue
        run_rows.extend(rows)
        case_ms = [r["total_ms"] for r in rows if r["total_ms"] is not None]
        run_summary_rows.append({
            "architecture": arch, "run": run, "wall_time_s": round(wall, 2),
            "total_judge_calls": int(sum((r["judge_calls"] or 0) for r in rows)),
            "mean_case_ms": round(float(np.mean(case_ms)), 1) if case_ms else None,
        })

df_stability_runs = pd.DataFrame(run_rows)
df_run_summary = pd.DataFrame(run_summary_rows)
if failures:
    print("run failures:", failures)
display(df_run_summary)

### 7a. Stability summary

Score stability and denominator (item-count) stability are **separate** concepts —
both are reported.

In [ ]:
by = df_stability_runs.groupby(["benchmark_id", "architecture"]).agg(
    case_id=("case_id", "first"), gt_score=("gt_score", "first"),
    mean_score=("score", "mean"), score_std=("score", "std"),
    min_score=("score", "min"), max_score=("score", "max"),
    mean_item_count=("item_count", "mean"), item_count_std=("item_count", "std"),
    min_item_count=("item_count", "min"), max_item_count=("item_count", "max"),
).reset_index()
by["score_range"] = by["max_score"] - by["min_score"]
by["item_count_range"] = by["max_item_count"] - by["min_item_count"]
df_stability_by_case = by

stab = df_stability_by_case.groupby("architecture").agg(
    cases=("benchmark_id", "nunique"),
    mean_score_std=("score_std", "mean"), median_score_std=("score_std", "median"),
    mean_score_range=("score_range", "mean"), max_score_range=("score_range", "max"),
    mean_item_count_std=("item_count_std", "mean"),
    median_item_count_std=("item_count_std", "median"),
    mean_item_count_range=("item_count_range", "mean"),
    max_item_count_range=("item_count_range", "max"),
).reset_index()
stab["runs"] = N_RUNS

rs = df_run_summary.groupby("architecture").agg(
    mean_wall_time_s=("wall_time_s", "mean"),
    total_judge_calls=("total_judge_calls", "sum"),
    mean_case_latency_ms=("mean_case_ms", "mean"),
).reset_index()

df_stability_summary = stab.merge(rs, on="architecture")[[
    "architecture", "cases", "runs", "mean_score_std", "median_score_std",
    "mean_score_range", "max_score_range", "mean_item_count_std",
    "median_item_count_std", "mean_item_count_range", "max_item_count_range",
    "mean_wall_time_s", "total_judge_calls", "mean_case_latency_ms"]].round(4)
display(df_stability_summary)

## 8. Ground Truth Closeness

GT closeness uses each case's **3-run mean** score (not the individual runs as
independent GT cases). Errors are on the 0-1 scale; percentage-point columns are
added only for readability. `1 - MAE` is **not** called "accuracy".

In [ ]:
def _closeness(sub):
    s = sub.dropna(subset=["mean_score", "gt_score"])
    err = s["mean_score"] - s["gt_score"]
    return pd.Series({
        "MAE": err.abs().mean(),
        "bias": err.mean(),
        "RMSE": float((err ** 2).mean() ** 0.5),
        "max_abs_error": err.abs().max(),
    })


df_gt_closeness = (
    df_stability_by_case.groupby("architecture").apply(_closeness).reset_index()
)
for col in ["MAE", "bias", "RMSE", "max_abs_error"]:
    df_gt_closeness[col + "_pct_points"] = (df_gt_closeness[col] * 100).round(2)
df_gt_closeness[["MAE", "bias", "RMSE", "max_abs_error"]] = (
    df_gt_closeness[["MAE", "bias", "RMSE", "max_abs_error"]].round(4)
)
display(df_gt_closeness)

### 8a. Relative quality improvement

DAG's relative MAE improvement over G-Eval, computed **from these runs** (the old
~19% figure is not hard-coded — DAG extraction semantics changed).

In [ ]:
mae = df_gt_closeness.set_index("architecture")["MAE"]
g_eval_mae = mae.get("g_eval")
dag_mae = mae.get("dag")

if g_eval_mae and g_eval_mae > 0 and dag_mae is not None:
    dag_mae_rel_improvement = (g_eval_mae - dag_mae) / g_eval_mae
    print(f"g_eval MAE: {g_eval_mae:.4f}")
    print(f"dag MAE:    {dag_mae:.4f}")
    print(f"DAG has {dag_mae_rel_improvement * 100:.1f}% lower MAE than G-Eval on this benchmark.")
else:
    print("g_eval MAE is zero or unavailable; relative improvement is undefined.")

## 9. Per-Case Comparison

Sign convention for `difference_in_abs_error = g_eval_abs_error - dag_abs_error`:
**positive → DAG closer to GT**, **negative → G-Eval closer to GT**.
`architecture_score_difference = dag_mean_score - g_eval_mean_score`.

In [ ]:
piv = df_stability_by_case.set_index(["benchmark_id", "architecture"])


def _get(bid, arch, col):
    try:
        return piv.loc[(bid, arch), col]
    except KeyError:
        return None


rows = []
for bid in df_stability_by_case["benchmark_id"].unique():
    gt = _get(bid, "g_eval", "gt_score")
    if gt is None:
        gt = _get(bid, "dag", "gt_score")
    ge_mean = _get(bid, "g_eval", "mean_score")
    dg_mean = _get(bid, "dag", "mean_score")
    ge_abs = abs(ge_mean - gt) if (gt is not None and ge_mean is not None) else None
    dg_abs = abs(dg_mean - gt) if (gt is not None and dg_mean is not None) else None
    rows.append({
        "benchmark_id": bid,
        "case_id": _get(bid, "g_eval", "case_id") or _get(bid, "dag", "case_id"),
        "gt_score": gt,
        "g_eval_mean_score": ge_mean,
        "g_eval_score_std": _get(bid, "g_eval", "score_std"),
        "g_eval_score_range": _get(bid, "g_eval", "score_range"),
        "g_eval_item_count_mean": _get(bid, "g_eval", "mean_item_count"),
        "g_eval_item_count_range": _get(bid, "g_eval", "item_count_range"),
        "g_eval_abs_error": ge_abs,
        "dag_mean_score": dg_mean,
        "dag_score_std": _get(bid, "dag", "score_std"),
        "dag_score_range": _get(bid, "dag", "score_range"),
        "dag_item_count_mean": _get(bid, "dag", "mean_item_count"),
        "dag_item_count_range": _get(bid, "dag", "item_count_range"),
        "dag_abs_error": dg_abs,
        "difference_in_abs_error": (ge_abs - dg_abs) if (ge_abs is not None and dg_abs is not None) else None,
        "architecture_score_difference": (dg_mean - ge_mean) if (dg_mean is not None and ge_mean is not None) else None,
    })

df_case_comparison = pd.DataFrame(rows)
display(df_case_comparison)

### 9a. Win / loss / tie counts

Positive `difference_in_abs_error` = DAG closer; negative = G-Eval closer; within
`1e-9` = tie. Median per-case absolute error adds context beyond MAE.

In [ ]:
TOL = 1e-9
valid = df_case_comparison.dropna(subset=["difference_in_abs_error"])

dag_wins = int((valid["difference_in_abs_error"] > TOL).sum())
g_eval_wins = int((valid["difference_in_abs_error"] < -TOL).sum())
ties = int((valid["difference_in_abs_error"].abs() <= TOL).sum())

print(f"DAG closer to GT on {dag_wins} cases")
print(f"G-Eval closer to GT on {g_eval_wins} cases")
print(f"Tie on {ties} cases")
print(f"median |error| g_eval: {valid['g_eval_abs_error'].median():.4f}")
print(f"median |error| dag:    {valid['dag_abs_error'].median():.4f}")

## 10. Largest Disagreements

Top 10 cases by absolute gap between DAG and G-Eval mean scores — useful for
inspecting whether semantic consolidation drives particular failures.

In [ ]:
tmp = df_case_comparison.copy()
tmp["score_gap"] = (tmp["dag_mean_score"] - tmp["g_eval_mean_score"]).abs()

df_largest_disagreements = tmp.sort_values("score_gap", ascending=False).head(10)[[
    "benchmark_id", "case_id", "gt_score", "g_eval_mean_score", "dag_mean_score",
    "architecture_score_difference", "g_eval_abs_error", "dag_abs_error",
    "difference_in_abs_error", "g_eval_score_std", "dag_score_std",
    "g_eval_item_count_mean", "dag_item_count_mean",
]]
display(df_largest_disagreements)

## 11. Summary

Interpret the tables above; conclusions are **not** hard-coded — read them off the
latest run.

**Quality (GT closeness, §8):**
- lower **MAE** is better; lower **RMSE** is better; **|bias|** closer to zero is better.
- §8a gives DAG's relative MAE improvement over G-Eval on this dataset.

**Stability (§7a):**
- lower **score std / range** = more stable scoring.
- lower **item-count std / range** = more stable denominator construction.

**Operations (§6):**
- lower **wall time** is better; fewer **judge calls** is cheaper/faster.

**DAG semantic target (§5a / §5b):**
- item counts should generally cluster around ~10; **>10 is allowed**; **>20** may
  trigger safety batching (each such case shown in §5b).

**Decision guidance (no universal threshold):**
- If DAG has meaningfully lower MAE while latency/call cost stays acceptable →
  prefer **DAG**.
- If GT quality is effectively tied and G-Eval is materially cheaper/faster →
  **G-Eval** may be preferred for latency-sensitive use cases.

> Historical note: earlier revisions of this notebook labelled the one-call mode
> `one_call` / `source_coverage`; the current public naming is `g_eval` with metric
> name `coverage`.

## Appendix A — Optional: inspect one DAG case in verbose mode

Not part of the benchmark and **not auto-run** (verbose increases response
size/cost). Run manually to see item-level reasons for a single case.

In [ ]:
# Optional single-case diagnostic — run manually.
verbose_dag = CoverageEvaluator(judge, mode="dag", verbose=True)
verbose_framework = EvaluationFramework(judge=judge, evaluators=[verbose_dag])

vr = verbose_framework.evaluate(cases[0])["coverage"]
print("score:", vr.score, "| label:", vr.label)
for item in (vr.details or {}).get("items", []):
    print(item)

## Appendix B — Optional: grouped shared-extraction smoke

Not part of the GT quality comparison. Demonstrates the DAG grouped-reuse path
(one shared context, multiple outputs → **one** extraction reused for each output's
classification) on a small synthetic group. Run manually.

In [ ]:
# Optional grouped-reuse smoke — run manually.
group = {
    "context": {"theme_description": "Shared source used for the grouped-reuse smoke."},
    "outputs": [{"epic_description": "First generated output."},
                {"epic_description": "Second generated output."}],
    "group_id": "smoke",
}
grouped_results = dag_framework.evaluate_groups([group])
for result_map in grouped_results:
    d = result_map["coverage"].details or {}
    print("shared_extraction:", d.get("shared_extraction"),
          "| classification_calls:", d.get("classification_calls"),
          "| judge_call_count:", d.get("judge_call_count"))

## Appendix C — Optional: Direct Azure vs Corporate Gateway (diagnostic)

Isolates gateway overhead vs model/prompt latency by running the same coverage case
through both judge paths. Diagnostic only, **not auto-run**. `AzureOpenAIJudge` and
`PROXY_URL` come from your existing session/config.

In [ ]:
row = df.iloc[0]
case = EvaluationCase(
    context={
        "theme_business_needs": row.get("theme_businessNeeds", ""),
        "theme_description": row.get("theme_description", ""),
    },
    output={
        "epic_description": row.get("epic_description", ""),
        "epic_success_criteria": row.get("epic_successCriteria", ""),
    },
    case_id=str(row.get("epic_key", 0)),
)

# A. corporate gateway (reuse the existing framework/judge)
start = time.perf_counter()
try:
    gateway_result = g_eval_framework.evaluate(case)["coverage"]
    gateway_elapsed = time.perf_counter() - start
    print(f"Gateway wall time: {gateway_elapsed:.2f}s | score {gateway_result.score} | {gateway_result.label}")
except Exception as exc:
    gateway_elapsed = time.perf_counter() - start
    gateway_result = None
    print(f"Gateway failed after {gateway_elapsed:.2f}s: {type(exc).__name__}: {exc}")

# B. direct Azure (reuse the existing notebook-local AzureOpenAIJudge)
azure_judge = AzureOpenAIJudge(timeout=90.0, verify_ssl=False, proxy_url=PROXY_URL)
azure_framework = EvaluationFramework(
    judge=azure_judge,
    evaluators=[CoverageEvaluator(azure_judge, mode="g_eval", verbose=False)],
)

start = time.perf_counter()
try:
    azure_result = azure_framework.evaluate(case)["coverage"]
    azure_elapsed = time.perf_counter() - start
    print(f"Direct Azure wall time: {azure_elapsed:.2f}s | score {azure_result.score} | {azure_result.label}")
except Exception as exc:
    azure_elapsed = time.perf_counter() - start
    azure_result = None
    print(f"Direct Azure failed after {azure_elapsed:.2f}s: {type(exc).__name__}: {exc}")

df_gateway_vs_azure = pd.DataFrame([
    {"path": "corporate_gateway", "success": gateway_result is not None,
     "wall_time_s": round(gateway_elapsed, 2),
     "score": gateway_result.score if gateway_result else None,
     "label": gateway_result.label if gateway_result else None},
    {"path": "direct_azure", "success": azure_result is not None,
     "wall_time_s": round(azure_elapsed, 2),
     "score": azure_result.score if azure_result else None,
     "label": azure_result.label if azure_result else None},
])
display(df_gateway_vs_azure)

### How to read the direct-Azure result

- **Direct Azure much faster than gateway** — gateway overhead is the main bottleneck.
- **Direct Azure succeeds but > 60s** — the model/prompt is slow; the gateway timeout
  simply cuts it off first.
- **Direct Azure also fails / approaches 90s** — the coverage request itself is heavy.
- **Both complete quickly** — the timeout was likely transient.